# Curriculum 01 · Lab 5 — Semantic Splitting

**Goal:** Compare *semantic* chunking against *recursive character* chunking on two real long-prose documents — the opening of *Pride and Prejudice* and of *Moby-Dick* from the Gutenberg corpus (`Data/corpus/gutenberg/`).

```
Recursive splitter : cuts at a fixed character count (500 chars, 50 overlap)
Semantic splitter  : embeds every sentence (local BGE) and draws a chunk
                     boundary exactly where the *meaning* shifts — where the
                     embedding distance between consecutive sentences jumps
                     above the 95th-percentile of all such distances
Embedding          : BAAI/bge-base-en-v1.5 (sentence-transformers, local)
```

Each book is bounded to its first `SUBSET_CHARS` (50K) characters: semantic splitting embeds every sentence locally, so a full ~700KB novel would be far too slow — 50K characters of prose is roughly 1,300 sentences and keeps one run in the low minutes.

**What to look for in the output:**

* the semantic splitter produces more, smaller, topically coherent chunks — roughly one per chapter of each novel;
* every boundary it draws sits at a sentence-level cosine-distance spike that clears the 95th-percentile threshold;
* the recursive splitter's 500-char chunks are blind to those topic boundaries and cut mid-sentence.

## 0 · Setup — dependencies & repo-root imports

Before running:

* **`sentence-transformers`** runs the local BGE embeddings (`BAAI/bge-base-en-v1.5`) — install cell below. The model is already cached in `~/.cache/huggingface`, so it loads from disk without a download — but the run takes ~2–5 minutes, let it finish.
* **Run this notebook from the repo root** — same as `python curriculum/01-chunking/05-semantic.py`. The first code cell locates the repo root (walking up from the working directory, since some runners start the kernel in the notebook's folder) and switches to it, so every repo-root-relative data path behaves exactly like the `.py` lab.
* `numpy` and `langchain-core` come from `requirements.txt`; the lab's component imports (`loaders.gutenberg`, `splitters.recursive`, `splitters.semantic`) come from the repo-root library.

In [1]:
# Needed for THIS project only:
#   sentence-transformers -> local BGE embeddings (BAAI/bge-base-en-v1.5)
#   (numpy, langchain-core are already in requirements.txt)
%pip install sentence-transformers


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
from langchain_core.documents import Document

# Make the repo-root component library importable. Unlike the .py lab there
# is no ``__file__`` to anchor the repo root, and some runners (nbconvert)
# start the kernel in the notebook's own directory, so the repo root is
# located by walking up from the current working directory and then switched
# to — this makes every repo-root-relative path behave exactly like the .py
# lab run from the repo root.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "loaders" / "gutenberg.py").is_file():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError(f"cannot locate repo root (loaders/gutenberg.py) from {Path.cwd()}")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from loaders.gutenberg import GutenbergLoader  # noqa: E402
from splitters.recursive import DocumentProcessor  # noqa: E402
from splitters.semantic import _SENTENCE_RE, SemanticSplitter  # noqa: E402

## 1 · Constants — chunking configuration

All lab knobs live at module level, exactly as in `curriculum/01-chunking/05-semantic.py`:

* `MODEL_NAME` — local BGE model (`splitters/semantic.py` default);
* `BREAKPOINT_PERCENTILE` — distance percentile above which a boundary is drawn (95th);
* `CHUNK_SIZE` / `CHUNK_OVERLAP` — the recursive splitter's fixed character budget;
* `SUBSET_CHARS` — per-book prefix cap that keeps sentence-by-sentence BGE embedding within a sane runtime;
* `DISTANCE_PRINT_CAP` — caps the printed distance array so a ~1300-sentence book doesn't flood the output;
* `DOC_PATHS` — the two Gutenberg novels (relative to the repo root);
* `PREVIEW` / `SENTENCE_PREVIEW` — character caps for chunk and boundary-sentence previews.

In [3]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"  # local BGE model (splitters/semantic.py default)
BREAKPOINT_PERCENTILE = 95.0  # distance percentile above which a boundary is drawn
CHUNK_SIZE = 500  # recursive splitter: max characters per chunk
CHUNK_OVERLAP = 50  # recursive splitter: characters shared across chunks
SUBSET_CHARS = 50_000  # per-book prefix cap (keeps sentence embedding runtime sane)
DISTANCE_PRINT_CAP = 40  # max consecutive-sentence distances printed per doc
DOC_PATHS = [
    Path("Data/corpus/gutenberg/pride-and-prejudice.txt"),
    Path("Data/corpus/gutenberg/moby-dick.txt"),
]
PREVIEW = 200  # character cap for chunk previews
SENTENCE_PREVIEW = 80  # character cap for boundary-neighbour sentences

## 2 · Load — Gutenberg novels bounded to `SUBSET_CHARS`

`GutenbergLoader` strips the Project Gutenberg header/footer, then each book's text is sliced to the first `SUBSET_CHARS` characters so the sentence-by-sentence BGE embedding stays within a sane runtime. Each document carries its source path as metadata — that metadata is how the per-book chunk counts below are attributed.

In [4]:
def load_books(paths: list[Path]) -> list[Document]:
    """Load Gutenberg novels, boilerplate stripped, each bounded to a prefix.

    ``GutenbergLoader`` removes the Project Gutenberg header/footer, then
    each book's text is sliced to the first ``SUBSET_CHARS`` characters so
    the sentence-by-sentence BGE embedding stays within a sane runtime.
    """
    docs: list[Document] = []
    for path in paths:
        text = GutenbergLoader(path, strip=True).load()[0].page_content
        docs.append(
            Document(
                page_content=text[:SUBSET_CHARS],
                metadata={"source": str(path)},
            )
        )
    return docs

## 3 · Boundary inspection — the teaching point

These three helpers make the semantic splitter's decisions visible:

* `consecutive_distances` — cosine distance between consecutive sentences (1 − cosine similarity), mirroring the internals of `splitters/semantic.py` so the numbers printed here are exactly the ones the `SemanticSplitter` used to draw boundaries;
* `boundary_pairs` — the first `limit` boundaries as `(pair_no, index, distance, prev, next)`, i.e. every sentence pair whose distance clears the threshold;
* `preview` — collapses whitespace and caps a sentence/chunk preview at `limit` characters.

In [5]:
def consecutive_distances(model, sentences: list[str]) -> np.ndarray:
    """Cosine distance between consecutive sentences (1 - cosine similarity).

    Mirrors the internals of ``splitters/semantic.py`` so the numbers printed
    here are exactly the ones the SemanticSplitter used to draw boundaries.
    """
    vectors = np.asarray(model.encode(sentences), dtype="float32")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0  # guard against zero-vector sentences
    normalized = vectors / norms
    dots = np.sum(normalized[:-1] * normalized[1:], axis=1)
    return 1.0 - np.clip(dots, -1.0, 1.0)


def boundary_pairs(
    sentences: list[str], distances: np.ndarray, threshold: float, limit: int = 3
) -> list[tuple[int, int, float, str, str]]:
    """First ``limit`` boundaries as (pair_no, index, distance, prev, next)."""
    pairs: list[tuple[int, int, float, str, str]] = []
    boundary_count = 0
    for i, distance in enumerate(distances):
        if distance > threshold:
            boundary_count += 1
            pairs.append(
                (boundary_count, i, float(distance), sentences[i], sentences[i + 1])
            )
            if len(pairs) == limit:
                break
    return pairs


def preview(text: str, limit: int) -> str:
    """Collapse whitespace and cap a sentence/chunk preview at ``limit`` chars."""
    flat = " ".join(text.split())
    return flat if len(flat) <= limit else flat[:limit] + "..."

## 4 · Comparison helper

`avg_words` — average word count across chunks, used to show that semantic chunks are smaller and more numerous while recursive chunks are uniform 500-char blocks.

In [6]:
def avg_words(chunks: list[Document]) -> float:
    """Average word count across chunks (0.0 for an empty chunk list)."""
    if not chunks:
        return 0.0
    return sum(len(c.page_content.split()) for c in chunks) / len(chunks)

## 5 · Load & split — recursive baseline vs semantic splitter

Now the actual run. The `sentence-transformers` guard prints a SKIP hint instead of failing if the package is missing. One local BGE model is shared by the splitter **and** the boundary analysis, so both operate on identical sentence embeddings.

* `DocumentProcessor` (from `splitters/recursive.py`) — the recursive character baseline;
* `SemanticSplitter` (from `splitters/semantic.py`) — splits `docs` with the model.

In [7]:
docs = load_books(DOC_PATHS)
print(
    f"Loaded {len(docs)} Gutenberg books (each limited to the first "
    f"{SUBSET_CHARS} characters):"
)
for doc in docs:
    print(f"  {doc.metadata['source']} — {len(doc.page_content)} chars")

Loaded 2 Gutenberg books (each limited to the first 50000 characters):
  Data/corpus/gutenberg/pride-and-prejudice.txt — 50000 chars
  Data/corpus/gutenberg/moby-dick.txt — 50000 chars


In [8]:
try:
    import sentence_transformers
except ImportError:
    print(
        "SKIP: semantic chunking needs sentence-transformers: "
        "pip install sentence-transformers"
    )
    semantic_available = False
else:
    semantic_available = True

# Recursive baseline: fixed character budget, blind to meaning.
recursive = DocumentProcessor(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
recursive_chunks = recursive.split_docs(docs)
print(
    f"Recursive split (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}): "
    f"{len(recursive_chunks)} chunk(s), avg {avg_words(recursive_chunks):.1f} words/chunk"
)

Recursive split (chunk_size=500, chunk_overlap=50): 257 chunk(s), avg 63.3 words/chunk


In [9]:
if semantic_available:
    # One local BGE model shared by the splitter and the boundary analysis,
    # so both operate on identical sentence embeddings.
    model = sentence_transformers.SentenceTransformer(MODEL_NAME)
    semantic = SemanticSplitter(
        embedding=model, breakpoint_percentile=BREAKPOINT_PERCENTILE
    )
    semantic_chunks = semantic.split(docs)
    print(
        f"Semantic split (breakpoint_percentile={BREAKPOINT_PERCENTILE:.0f}): "
        f"{len(semantic_chunks)} chunk(s), avg {avg_words(semantic_chunks):.1f} words/chunk"
    )
    for doc in docs:
        source = doc.metadata["source"]
        n_recursive = sum(
            1 for c in recursive_chunks if c.metadata.get("source") == source
        )
        n_semantic = sum(1 for c in semantic_chunks if c.metadata.get("source") == source)
        print(f"  {source}: {n_recursive} recursive chunk(s) vs {n_semantic} semantic chunk(s)")

Semantic split (breakpoint_percentile=95): 52 chunk(s), avg 311.1 words/chunk
  Data/corpus/gutenberg/pride-and-prejudice.txt: 122 recursive chunk(s) vs 18 semantic chunk(s)
  Data/corpus/gutenberg/moby-dick.txt: 135 recursive chunk(s) vs 34 semantic chunk(s)


## 6 · Teaching point — boundaries sit at cosine-distance spikes

For each book: split the prose into sentences, embed them, and compute consecutive-sentence distances. The printed distance array marks every boundary the splitter drew with `*` — the spike pattern is visible at a glance (capped at `DISTANCE_PRINT_CAP` so a ~1300-sentence book doesn't flood output). Each of the first `limit` boundaries is then shown with its distance, how far above the p95 threshold it spikes, how many times the mean distance it is, and the two sentences the boundary falls between.

In [10]:
print("\n--- Where does the semantic splitter draw boundaries? ---")
for doc in docs:
    sentences = [
        s for s in _SENTENCE_RE.split(doc.page_content.strip()) if s
    ]
    if len(sentences) < 2:
        print(f"{doc.metadata['source']}: too few sentences to analyse")
        continue
    distances = consecutive_distances(model, sentences)
    threshold = float(np.percentile(distances, BREAKPOINT_PERCENTILE))
    print(
        f"\n{doc.metadata['source']} — {len(sentences)} sentence(s), "
        f"{len(distances)} consecutive-sentence distances"
    )
    print(
        f"  distance stats: mean={distances.mean():.3f}  "
        f"max={distances.max():.3f}  "
        f"p{BREAKPOINT_PERCENTILE:.0f} threshold={threshold:.3f}"
    )
    # The distance array with '*' marking every boundary the splitter
    # drew: the spike pattern is visible at a glance. Capped at
    # DISTANCE_PRINT_CAP so a ~1300-sentence book doesn't flood output.
    shown = distances[:DISTANCE_PRINT_CAP]
    marked = " ".join(
        f"{d:.2f}" + ("*" if d > threshold else "") for d in shown
    )
    n_more = len(distances) - len(shown)
    more_note = f" ... ({n_more} more)" if n_more > 0 else ""
    print(
        f"  distances (first {len(shown)} of {len(distances)}): "
        f"{marked}{more_note}   (* = boundary, distance > p95 threshold)"
    )
    for pair_no, i, distance, prev_sentence, next_sentence in boundary_pairs(
        sentences, distances, threshold
    ):
        print(
            f"  chunk pair {pair_no} -> {pair_no + 1}: boundary after sentence "
            f"{i + 1}, distance {distance:.3f} > p95 threshold {threshold:.3f} "
            f"(spike +{distance - threshold:.3f}, "
            f"{distance / distances.mean():.2f}x the mean distance)"
        )
        print(f"    ends: {preview(prev_sentence, SENTENCE_PREVIEW)}")
        print(f"    next: {preview(next_sentence, SENTENCE_PREVIEW)}")


--- Where does the semantic splitter draw boundaries? ---



Data/corpus/gutenberg/pride-and-prejudice.txt — 335 sentence(s), 334 consecutive-sentence distances
  distance stats: mean=0.480  max=0.702  p95 threshold=0.621
  distances (first 40 of 334): 0.18 0.49 0.44 0.47 0.49 0.50 0.50 0.40 0.46 0.30 0.47 0.47 0.46 0.37 0.32 0.51 0.38 0.47 0.54 0.57 0.55 0.49 0.51 0.51 0.43 0.41 0.32 0.37 0.62* 0.50 0.30 0.34 0.54 0.36 0.58 0.47 0.41 0.51 0.52 0.42 ... (294 more)   (* = boundary, distance > p95 threshold)
  chunk pair 1 -> 2: boundary after sentence 29, distance 0.623 > p95 threshold 0.621 (spike +0.003, 1.30x the mean distance)
    ends: It sets off his other gifts and graces most advantageously to the critical eye; ...
    next: But a very badly-built novel which excelled in pathetic or humorous character, o...
  chunk pair 2 -> 3: boundary after sentence 83, distance 0.623 > p95 threshold 0.621 (spike +0.003, 1.30x the mean distance)
    ends: Collins is perfectly natural, and perfectly alive.
    next: In fact, for all the “miniature,” the


Data/corpus/gutenberg/moby-dick.txt — 648 sentence(s), 647 consecutive-sentence distances
  distance stats: mean=0.484  max=0.747  p95 threshold=0.605
  distances (first 40 of 647): 0.26 0.45 0.42 0.45 0.42 0.45 0.46 0.51 0.52 0.43 0.44 0.45 0.45 0.42 0.42 0.44 0.44 0.48 0.50 0.42 0.42 0.47 0.47 0.50 0.50 0.42 0.42 0.45 0.46 0.51 0.51 0.50 0.47 0.45 0.47 0.51 0.50 0.42 0.44 0.45 ... (607 more)   (* = boundary, distance > p95 threshold)
  chunk pair 1 -> 2: boundary after sentence 125, distance 0.627 > p95 threshold 0.605 (spike +0.021, 1.29x the mean distance)
    ends: CHAPTER 61.
    next: Stubb Kills a Whale.
  chunk pair 2 -> 3: boundary after sentence 149, distance 0.670 > p95 threshold 0.605 (spike +0.065, 1.39x the mean distance)
    ends: CHAPTER 73.
    next: Stubb and Flask kill a Right Whale; and Then Have a Talk over Him.
  chunk pair 3 -> 4: boundary after sentence 150, distance 0.654 > p95 threshold 0.605 (spike +0.049, 1.35x the mean distance)
    ends: Stubb and Flask 

## 7 · Content preview — semantic chunks group related sentences

Same prose excerpt, two splitters: the semantic chunk keeps a stretch of narrative (a chapter opening, a train of thought) whole because its sentences embed closely; the recursive chunk cuts at a fixed 500 characters and can land mid-sentence.

In [11]:
print("\n--- Content preview: semantic chunks group related sentences ---")
# Same prose excerpt, two splitters: the semantic chunk keeps a stretch
# of narrative (a chapter opening, a train of thought) whole because its
# sentences embed closely; the recursive chunk cuts at a fixed 500
# characters and can land mid-sentence.
first_semantic = (
    semantic_chunks[1] if len(semantic_chunks) > 1 else semantic_chunks[0]
)
first_recursive = (
    recursive_chunks[1] if len(recursive_chunks) > 1 else recursive_chunks[0]
)
print(
    f"Semantic chunk [1] ({avg_words([first_semantic]):.0f} words, "
    f"source {first_semantic.metadata['source']}):"
)
print(f"  {preview(first_semantic.page_content, PREVIEW)}")
print(
    f"Recursive chunk [1] ({avg_words([first_recursive]):.0f} words, "
    f"source {first_recursive.metadata['source']}):"
)
print(f"  {preview(first_recursive.page_content, PREVIEW)}")


--- Content preview: semantic chunks group related sentences ---
Semantic chunk [1] (1460 words, source Data/corpus/gutenberg/pride-and-prejudice.txt):
  But a very badly-built novel which excelled in pathetic or humorous character, or which displayed consummate command of dialogue--perhaps the rarest of all faculties--would be an infinitely better thi...
Recursive chunk [1] (19 words, source Data/corpus/gutenberg/pride-and-prejudice.txt):
  PRIDE. and PREJUDICE by Jane Austen, with a Preface by George Saintsbury and Illustrations by Hugh Thomson [Illustration: 1894]


## Takeaway

The semantic splitter draws a boundary at every sentence pair whose cosine distance clears the 95th-percentile threshold — exactly where the topic shifts — while the recursive splitter cuts at a fixed `CHUNK_SIZE` characters, blind to meaning.

In [12]:
print(
    "\nTeaching takeaway: the semantic splitter draws a boundary at every "
    "sentence pair whose\ncosine distance clears the "
    f"{BREAKPOINT_PERCENTILE:.0f}th-percentile threshold — exactly where "
    "the topic shifts —\nwhile the recursive splitter cuts at a fixed "
    f"{CHUNK_SIZE} characters, blind to meaning."
)


Teaching takeaway: the semantic splitter draws a boundary at every sentence pair whose
cosine distance clears the 95th-percentile threshold — exactly where the topic shifts —
while the recursive splitter cuts at a fixed 500 characters, blind to meaning.


## What you should notice

* **Counts:** semantic splitting yields more, smaller chunks per book (roughly one per chapter); recursive splitting yields uniform ~500-char blocks.
* **Boundaries:** every semantic boundary sits at a sentence-level cosine-distance spike above the p95 threshold; recursive cuts land mid-sentence.
* **Locality:** the whole lab embeds locally (`BAAI/bge-base-en-v1.5`) — no API calls.